# تست مدل اول ارتقا یافته

In [ ]:
pip install -q ultralytics opencv-python-headless transformers torch torchvision pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 7.5 MB/s eta 0:00:00


In [ ]:
# ============================================================
#  SECURITY PIPELINE v4 - POSE-BASED SMART FACE VISIBILITY
#  Single Model: YOLO11-pose (Person + 17 Keypoints + ByteTrack)
#  -> Face-Visibility via Eye/Nose keypoint confidence (no separate face model)
#  -> Auto Align+Crop via eye-line rotation -> SigLIP Mask Classifier
#  -> 4-State Machine: Gray(analyzing) / Green(locked) / Orange / Red(focus-loop)
# ============================================================
# !pip install -q ultralytics opencv-python-headless transformers torch torchvision pillow


import cv2, time, torch, numpy as np
from collections import deque
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, SiglipForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_HALF = device == "cuda"
print(f"🔧 Device: {device} | FP16: {USE_HALF}")

# ---------------- 1) Single Unified Model: Person+Pose+Track ----------
pose_model = YOLO("yolo11n-pose.pt")   # سبک، سریع، خودکار دانلود می‌شه

# COCO-17 keypoint indices مورد نیاز ما
NOSE, LEYE, REYE, LEAR, REAR = 0, 1, 2, 3, 4

# ---------------- 2) Mask Classifier (Pretrained, FP16) ----------------
MASK_MODEL_NAME = "prithivMLmods/Face-Mask-Detection"
mask_processor = AutoImageProcessor.from_pretrained(MASK_MODEL_NAME)
mask_model = SiglipForImageClassification.from_pretrained(MASK_MODEL_NAME).to(device).eval()
if USE_HALF:
    mask_model = mask_model.half()
MASK_ID2LABEL = {0: "mask", 1: "no_mask"}

@torch.no_grad()
def classify_mask_batch(face_list):
    if len(face_list) == 0:
        return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in face_list]
    inputs = mask_processor(images=pil_imgs, return_tensors="pt").to(device)
    if USE_HALF:
        inputs = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in inputs.items()}
    logits = mask_model(**inputs).logits.float()
    probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
    out = []
    for p in probs:
        pred = int(np.argmax(p))
        out.append((MASK_ID2LABEL[pred], float(p[pred])))
    return out

# ---------------- 3) Skin-ratio heuristic (Medical vs Suspicious) -----
def skin_ratio(region_bgr):
    if region_bgr is None or region_bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 20, 40], dtype=np.uint8)
    upper = np.array([25, 180, 255], dtype=np.uint8)
    m = cv2.inRange(hsv, lower, upper)
    return float(np.count_nonzero(m)) / m.size

def is_suspicious(face_bgr):
    h, w, _ = face_bgr.shape
    upper = face_bgr[0:int(h * 0.45), :]
    return skin_ratio(upper) < 0.12

# ---------------- 4) Keypoint-based Face Visibility + Align + Crop ----
def align_and_crop_face(person_img, kxy, kconf, conf_th=0.5):
    """
    برمی‌گرداند: (face_crop یا None, face_visibility_score)
    اگر نقاط چشم/بینی با اطمینان کافی دیده نشن (نیم‌رخ شدید/پشت به دوربین) -> None
    """
    nose_c, leye_c, reye_c = kconf[NOSE], kconf[LEYE], kconf[REYE]
    face_score = float((nose_c + leye_c + reye_c) / 3.0)
    if face_score < conf_th:
        return None, face_score

    leye, reye = kxy[LEYE], kxy[REYE]
    eye_dist = float(np.linalg.norm(np.array(leye) - np.array(reye)))
    if eye_dist < 3:
        return None, face_score

    h, w = person_img.shape[:2]
    dy, dx = reye[1] - leye[1], reye[0] - leye[0]
    angle = np.degrees(np.arctan2(dy, dx))
    eye_center = ((leye[0] + reye[0]) / 2.0, (leye[1] + reye[1]) / 2.0)

    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    rotated = cv2.warpAffine(person_img, M, (w, h))

    half_w = eye_dist * 1.6
    top = eye_center[1] - eye_dist * 1.3
    bottom = eye_center[1] + eye_dist * 2.6  # شامل بینی/دهان/چانه

    x1, x2 = int(max(0, eye_center[0] - half_w)), int(min(w, eye_center[0] + half_w))
    y1, y2 = int(max(0, top)), int(min(h, bottom))
    if (x2 - x1) < 10 or (y2 - y1) < 10:
        return None, face_score

    return rotated[y1:y2, x1:x2], face_score

# ---------------- 5) Smart State Manager (exact behavior table) -------
COLORS = {"gray": (160, 160, 160), "green": (0, 200, 0),
          "orange": (0, 140, 255), "red": (0, 0, 255)}
LABELS = {"gray": "Analyzing...", "green": "Clear",
          "orange": "Medical Mask", "red": "SUSPICIOUS - ALERT"}

class TrackStateManager:
    def __init__(self):
        self.data = {}
        self.FAST_VOTES_NEEDED = 3      # فرد جدید: تا ۳ رأی سریع تصمیم بگیر
        self.FOCUS_INTERVAL = 2         # حالت فوکوس (ماسک/مشکوک): هر ۲ فریم
        self.FOCUS_WINDOW = 4           # پنجره رأی‌گیری چرخشی در حالت فوکوس
        self.FOCUS_GREEN_NEEDED = 3     # چند رأی سبز لازم برای بازگشت به سبز

    def ensure(self, tid):
        if tid not in self.data:
            self.data[tid] = {
                "mode": "fast",          # fast | focus
                "locked": False,
                "votes": [],             # برای حالت fast
                "focus_window": deque(maxlen=self.FOCUS_WINDOW),
                "color": "gray", "label": LABELS["gray"]
            }
        return self.data[tid]

    def should_analyze(self, tid, frame_idx):
        st = self.data[tid]
        if st["locked"]:
            return False
        if st["mode"] == "fast":
            return True                      # هر فریم
        return frame_idx % self.FOCUS_INTERVAL == 0   # هر ۲ فریم

    def register_vote(self, tid, category, conf):
        st = self.data[tid]

        if st["mode"] == "fast":
            st["votes"].append((category, conf))
            if len(st["votes"]) >= self.FAST_VOTES_NEEDED:
                score = {"green": 0.0, "orange": 0.0, "red": 0.0}
                for c, cf in st["votes"]:
                    score[c] += cf
                best = max(score, key=score.get)
                if best == "green":
                    self._lock(tid, "green")     # ✅ سبز = قفل نهایی، توقف پردازش
                else:
                    # ماسک/مشکوک -> برو به حالت فوکوس، هرگز کاملا قفل نشو
                    st["mode"] = "focus"
                    st["color"], st["label"] = best, LABELS[best]
                    st["focus_window"].append(best)

        else:  # focus mode
            st["focus_window"].append(category)
            window = list(st["focus_window"])
            green_count = window.count("green")
            if green_count >= self.FOCUS_GREEN_NEEDED:
                self._lock(tid, "green")         # چهره واضح شد -> قفل نهایی سبز
            else:
                # وضعیت غالب بین orange/red رو نمایش بده (می‌تونه بین این دو نوسان کنه)
                sub = [c for c in window if c in ("orange", "red")]
                if sub:
                    best = max(set(sub), key=sub.count)
                    st["color"], st["label"] = best, LABELS[best]

    def _lock(self, tid, category):
        st = self.data[tid]
        st["locked"] = True
        st["color"] = category
        st["label"] = LABELS[category]

state_mgr = TrackStateManager()

# ---------------- 6) Main Pipeline --------------------------------------
def process_video(input_path, output_path, conf_thres=0.4, yolo_imgsz=640, face_conf_th=0.5):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Error: Cannot open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    t0 = time.time()
    last_pct = -1
    frame_idx = 0

    results_gen = pose_model.track(
        source=input_path, classes=[0], conf=conf_thres, imgsz=yolo_imgsz,
        tracker="bytetrack.yaml", stream=True, verbose=False, persist=True,
        half=USE_HALF
    )

    for r in results_gen:
        frame_idx += 1
        frame = r.orig_img

        if r.boxes.id is None or r.keypoints is None:
            out.write(frame)
            pct = int(frame_idx / total * 100) if total > 0 else 0
            if pct != last_pct and pct % 5 == 0:
                print(f"⏳ Progress: {pct}%"); last_pct = pct
            continue

        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        kpts_xy_all = r.keypoints.xy.cpu().numpy()
        kpts_conf_all = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(kpts_xy_all.shape[:2])

        batch_crops, batch_meta = [], []

        for tid, box, kxy, kconf in zip(ids, boxes, kpts_xy_all, kpts_conf_all):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            person_crop = frame[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue

            st = state_mgr.ensure(tid)

            if state_mgr.should_analyze(tid, frame_idx):
                # کی‌پوینت‌ها نسبت به کل فریم هستن؛ به مختصات داخل person_crop منتقل می‌کنیم
                local_kxy = kxy.copy()
                local_kxy[:, 0] -= x1
                local_kxy[:, 1] -= y1

                face_crop, face_score = align_and_crop_face(person_crop, local_kxy, kconf, face_conf_th)

                if face_crop is not None:
                    batch_crops.append(face_crop)
                    batch_meta.append(tid)
                # اگر face_crop == None یعنی نیم‌رخ شدید/پشت به دوربین -> کاملا سایلنت، رأی ثبت نمی‌شه

        # ---- Batch Mask Classification ----
        if batch_crops:
            results = classify_mask_batch(batch_crops)
            for (tid, (mask_label, conf), face_bgr) in zip(batch_meta, results, batch_crops):
                if mask_label == "no_mask":
                    state_mgr.register_vote(tid, "green", conf)
                else:
                    cat = "red" if is_suspicious(face_bgr) else "orange"
                    state_mgr.register_vote(tid, cat, conf)

        # ---- رسم خروجی ----
        for tid, box in zip(ids, boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            st = state_mgr.ensure(tid)
            color = COLORS[st["color"]]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {tid}: {st['label']}", (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
            if st["color"] == "red":
                cv2.putText(frame, "ALERT!", (x1, y2 + 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        out.write(frame)

        pct = int(frame_idx / total * 100) if total > 0 else 0
        if pct != last_pct and pct % 5 == 0:
            print(f"⏳ Progress: {pct}%")
            last_pct = pct

    out.release()
    print(f"✅ Done in {time.time() - t0:.2f}s -> {output_path}")

# ---------------- 7) RUN --------------------------------------------------
# input_video_path = '/content/drive/My Drive/9.mp4'
input_video_path = "C:\1\1_پروژه\1_پروژه خودم\فیلم های پروژه بهینه شده\درگیری\9.mp4"
output_video_path = "/content/output_result_(final).mp4"

process_video(input_video_path, output_video_path, conf_thres=0.4, yolo_imgsz=640, face_conf_th=0.5)

<>:271: SyntaxWarning: invalid escape sequence '\9'
<>:271: SyntaxWarning: invalid escape sequence '\9'
/tmp/ipykernel_2373/372542999.py:271: SyntaxWarning: invalid escape sequence '\9'
  input_video_path = "C:\1\1_پروژه\1_پروژه خودم\فیلم های پروژه بهینه شده\درگیری\9.mp4"


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
🔧 Device: cuda | FP16: True


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B /  372MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

❌ Error: Cannot open video


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# %pip install -q ultralytics opencv-python-headless
%pip install transformers torch torchvision pillow


In [ ]:
# ============================================================
#  SECURITY PIPELINE - PRESENTATION / DEMO VERSION
#  Same core logic as v4 (YOLO11-pose + ByteTrack + SigLIP mask classifier)
#  + Corner Picture-in-Picture gallery of detected people
#  + Dramatic Slow-Motion effect on new detection / RED alert
#  + Upper-body skeleton overlay for visual "wow" effect
# ============================================================

# !pip install -q ultralytics opencv-python-headless transformers torch torchvision pillow

import cv2, time, torch, numpy as np
from collections import deque
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, SiglipForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_HALF = device == "cuda"
print(f"🔧 Device: {device} | FP16: {USE_HALF}")

# ---------------- 1) Single Unified Model: Person+Pose+Track ----------
pose_model = YOLO("yolo11n-pose.pt")

NOSE, LEYE, REYE, LEAR, REAR = 0, 1, 2, 3, 4
LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST = 5, 6, 7, 8, 9, 10

# اسکلت فقط بالاتنه (سر، شونه، بازو، ساعد) - برای جلوه بصری
UPPER_BODY_SKELETON = [
    (LEYE, REYE), (NOSE, LEYE), (NOSE, REYE),
    (LEAR, LEYE), (REAR, REYE),
    (LSHOULDER, RSHOULDER),
    (LSHOULDER, LELBOW), (LELBOW, LWRIST),
    (RSHOULDER, RELBOW), (RELBOW, RWRIST),
    (NOSE, LSHOULDER), (NOSE, RSHOULDER),
]

# ---------------- 2) Mask Classifier (Pretrained, FP16) ----------------
MASK_MODEL_NAME = "prithivMLmods/Face-Mask-Detection"
mask_processor = AutoImageProcessor.from_pretrained(MASK_MODEL_NAME)
mask_model = SiglipForImageClassification.from_pretrained(MASK_MODEL_NAME).to(device).eval()
if USE_HALF:
    mask_model = mask_model.half()
MASK_ID2LABEL = {0: "mask", 1: "no_mask"}

@torch.no_grad()
def classify_mask_batch(face_list):
    if len(face_list) == 0:
        return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in face_list]
    inputs = mask_processor(images=pil_imgs, return_tensors="pt").to(device)
    if USE_HALF:
        inputs = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in inputs.items()}
    logits = mask_model(**inputs).logits.float()
    probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
    out = []
    for p in probs:
        pred = int(np.argmax(p))
        out.append((MASK_ID2LABEL[pred], float(p[pred])))
    return out

# ---------------- 3) Skin-ratio heuristic (Medical vs Suspicious) -----
def skin_ratio(region_bgr):
    if region_bgr is None or region_bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 20, 40], dtype=np.uint8)
    upper = np.array([25, 180, 255], dtype=np.uint8)
    m = cv2.inRange(hsv, lower, upper)
    return float(np.count_nonzero(m)) / m.size

def is_suspicious(face_bgr):
    h, w, _ = face_bgr.shape
    upper = face_bgr[0:int(h * 0.45), :]
    return skin_ratio(upper) < 0.12

# ---------------- 4) Keypoint-based Face Visibility + Align + Crop ----
def align_and_crop_face(person_img, kxy, kconf, conf_th=0.5):
    nose_c, leye_c, reye_c = kconf[NOSE], kconf[LEYE], kconf[REYE]
    face_score = float((nose_c + leye_c + reye_c) / 3.0)
    if face_score < conf_th:
        return None, face_score

    leye, reye = kxy[LEYE], kxy[REYE]
    eye_dist = float(np.linalg.norm(np.array(leye) - np.array(reye)))
    if eye_dist < 3:
        return None, face_score

    h, w = person_img.shape[:2]
    dy, dx = reye[1] - leye[1], reye[0] - leye[0]
    angle = np.degrees(np.arctan2(dy, dx))
    eye_center = ((leye[0] + reye[0]) / 2.0, (leye[1] + reye[1]) / 2.0)

    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    rotated = cv2.warpAffine(person_img, M, (w, h))

    half_w = eye_dist * 1.6
    top = eye_center[1] - eye_dist * 1.3
    bottom = eye_center[1] + eye_dist * 2.6

    x1, x2 = int(max(0, eye_center[0] - half_w)), int(min(w, eye_center[0] + half_w))
    y1, y2 = int(max(0, top)), int(min(h, bottom))
    if (x2 - x1) < 10 or (y2 - y1) < 10:
        return None, face_score

    return rotated[y1:y2, x1:x2], face_score

# ---------------- 5) Smart State Manager (same logic as v4) -----------
COLORS = {"gray": (160, 160, 160), "green": (0, 200, 0),
          "orange": (0, 140, 255), "red": (0, 0, 255)}
LABELS = {"gray": "Analyzing...", "green": "Clear",
          "orange": "Medical Mask", "red": "SUSPICIOUS - ALERT"}

class TrackStateManager:
    def __init__(self):
        self.data = {}
        self.FAST_VOTES_NEEDED = 3
        self.FOCUS_INTERVAL = 2
        self.FOCUS_WINDOW = 4
        self.FOCUS_GREEN_NEEDED = 3

    def ensure(self, tid):
        if tid not in self.data:
            self.data[tid] = {
                "mode": "fast", "locked": False, "votes": [],
                "focus_window": deque(maxlen=self.FOCUS_WINDOW),
                "color": "gray", "label": LABELS["gray"],
                "is_new": True,        # برای افکت نمایشی: آیا تازه معرفی شده؟
                "just_finalized": None # برای افکت نمایشی: آیا همین الان قفل نهایی گرفته؟
            }
        return self.data[tid]

    def should_analyze(self, tid, frame_idx):
        st = self.data[tid]
        if st["locked"]:
            return False
        if st["mode"] == "fast":
            return True
        return frame_idx % self.FOCUS_INTERVAL == 0

    def register_vote(self, tid, category, conf):
        st = self.data[tid]
        st["just_finalized"] = None

        if st["mode"] == "fast":
            st["votes"].append((category, conf))
            if len(st["votes"]) >= self.FAST_VOTES_NEEDED:
                score = {"green": 0.0, "orange": 0.0, "red": 0.0}
                for c, cf in st["votes"]:
                    score[c] += cf
                best = max(score, key=score.get)
                if best == "green":
                    self._lock(tid, "green")
                else:
                    st["mode"] = "focus"
                    st["color"], st["label"] = best, LABELS[best]
                    st["focus_window"].append(best)
                    if best == "red":
                        st["just_finalized"] = "red"
        else:
            st["focus_window"].append(category)
            window = list(st["focus_window"])
            green_count = window.count("green")
            if green_count >= self.FOCUS_GREEN_NEEDED:
                self._lock(tid, "green")
            else:
                sub = [c for c in window if c in ("orange", "red")]
                if sub:
                    best = max(set(sub), key=sub.count)
                    if best == "red" and st["color"] != "red":
                        st["just_finalized"] = "red"
                    st["color"], st["label"] = best, LABELS[best]

    def _lock(self, tid, category):
        st = self.data[tid]
        st["locked"] = True
        st["color"] = category
        st["label"] = LABELS[category]

state_mgr = TrackStateManager()

# ---------------- 6) Presentation Helpers (Gallery + Skeleton) --------
class PresentationGallery:
    """گالری تصاویر کوچک در گوشه تصویر - آخرین افراد شناسایی‌شده"""
    def __init__(self, max_items=4, thumb_size=140):
        self.items = deque(maxlen=max_items)   # هر آیتم: (img, label, color)
        self.thumb_size = thumb_size

    def add(self, crop_bgr, label, color):
        if crop_bgr is None or crop_bgr.size == 0:
            return
        thumb = cv2.resize(crop_bgr, (self.thumb_size, self.thumb_size))
        self.items.append((thumb, label, color))

    def draw(self, frame):
        h, w = frame.shape[:2]
        pad = 10
        for i, (thumb, label, color) in enumerate(self.items):
            x2 = w - pad
            x1 = x2 - self.thumb_size
            y1 = pad + i * (self.thumb_size + 35)
            y2 = y1 + self.thumb_size
            if y2 > h:
                break
            frame[y1:y2, x1:x2] = thumb
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            cv2.putText(frame, label, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

gallery = PresentationGallery(max_items=4, thumb_size=140)

def draw_upper_skeleton(frame, kxy, kconf, color=(255, 255, 0), conf_th=0.4):
    """رسم اسکلت بالاتنه فقط - برای جلوه بصری خفن"""
    for a, b in UPPER_BODY_SKELETON:
        if kconf[a] < conf_th or kconf[b] < conf_th:
            continue
        pa = tuple(map(int, kxy[a]))
        pb = tuple(map(int, kxy[b]))
        cv2.line(frame, pa, pb, color, 2)
    for idx in [NOSE, LEYE, REYE, LEAR, REAR, LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST]:
        if kconf[idx] >= conf_th:
            p = tuple(map(int, kxy[idx]))
            cv2.circle(frame, p, 4, color, -1)

# ---------------- 7) Main Pipeline (with dramatic slow-mo writing) ----
def process_video_demo(input_path, output_path, conf_thres=0.4, yolo_imgsz=640,
                        face_conf_th=0.5, slowmo_repeat=6):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Error: Cannot open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    t0 = time.time()
    last_pct = -1
    frame_idx = 0
    seen_ids = set()   # برای تشخیص "فرد کاملا جدید" جهت افکت اسلوموشن

    results_gen = pose_model.track(
        source=input_path, classes=[0], conf=conf_thres, imgsz=yolo_imgsz,
        tracker="bytetrack.yaml", stream=True, verbose=False, persist=True,
        half=USE_HALF
    )

    for r in results_gen:
        frame_idx += 1
        frame = r.orig_img

        if r.boxes.id is None or r.keypoints is None:
            out.write(frame)
            pct = int(frame_idx / total * 100) if total > 0 else 0
            if pct != last_pct and pct % 5 == 0:
                print(f"⏳ Progress: {pct}%"); last_pct = pct
            continue

        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        kpts_xy_all = r.keypoints.xy.cpu().numpy()
        kpts_conf_all = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(kpts_xy_all.shape[:2])

        batch_crops, batch_meta = [], []
        trigger_slowmo = False   # آیا این فریم باید کند نمایش داده بشه؟

        for tid, box, kxy, kconf in zip(ids, boxes, kpts_xy_all, kpts_conf_all):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            person_crop = frame[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue

            st = state_mgr.ensure(tid)

            # --- افکت نمایشی: فرد کاملا جدید -> اسلوموشن + اضافه به گالری ---
            if tid not in seen_ids:
                seen_ids.add(tid)
                trigger_slowmo = True
                square_crop = person_crop.copy()
                gallery.add(square_crop, f"New Person ID {tid}", (255, 255, 0))

            if state_mgr.should_analyze(tid, frame_idx):
                local_kxy = kxy.copy()
                local_kxy[:, 0] -= x1
                local_kxy[:, 1] -= y1

                face_crop, face_score = align_and_crop_face(person_crop, local_kxy, kconf, face_conf_th)

                if face_crop is not None:
                    batch_crops.append(face_crop)
                    batch_meta.append(tid)

            # --- رسم اسکلت بالاتنه (جلوه بصری) ---
            local_kxy_full = kxy.copy()
            draw_upper_skeleton(frame, local_kxy_full, kconf)

        if batch_crops:
            results = classify_mask_batch(batch_crops)
            for (tid, (mask_label, conf), face_bgr) in zip(batch_meta, results, batch_crops):
                if mask_label == "no_mask":
                    state_mgr.register_vote(tid, "green", conf)
                else:
                    cat = "red" if is_suspicious(face_bgr) else "orange"
                    state_mgr.register_vote(tid, cat, conf)

        # ---- رسم باکس‌ها + تشخیص لحظه هشدار قرمز برای اسلوموشن و گالری ----
        for tid, box in zip(ids, boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            st = state_mgr.ensure(tid)
            color = COLORS[st["color"]]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {tid}: {st['label']}", (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            if st["color"] == "red":
                cv2.putText(frame, "ALERT!", (x1, y2 + 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            if st.get("just_finalized") == "red":
                trigger_slowmo = True
                x1c, y1c = max(0, x1), max(0, y1)
                x2c, y2c = min(W, x2), min(H, y2)
                crop = frame[y1c:y2c, x1c:x2c].copy()
                gallery.add(crop, f"SUSPECT ID {tid}", (0, 0, 255))
                st["just_finalized"] = None  # فقط یکبار افکت رو نشون بده

        # ---- گالری گوشه تصویر ----
        gallery.draw(frame)

        # ---- نوشتن خروجی (با افکت اسلوموشن در لحظات کلیدی) ----
        repeat = slowmo_repeat if trigger_slowmo else 1
        for _ in range(repeat):
            out.write(frame)

        pct = int(frame_idx / total * 100) if total > 0 else 0
        if pct != last_pct and pct % 5 == 0:
            print(f"⏳ Progress: {pct}%")
            last_pct = pct

    out.release()
    print(f"✅ Demo video ready in {time.time() - t0:.2f}s -> {output_path}")

# ---------------- 8) RUN --------------------------------------------------
input_video_path = '/content/drive/My Drive/9.mp4'
# input_video_path = "C:\1\1_پروژه\1_پروژه خودم\فیلم های پروژه بهینه شده\درگیری\9.mp4"

output_video_path = "/content/output_demo_(Final).mp4"

process_video_demo(input_video_path, output_video_path,
                    conf_thres=0.4, yolo_imgsz=640, face_conf_th=0.5, slowmo_repeat=6)

🔧 Device: cuda | FP16: True


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 252ms
Prepared 1 package in 40ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
⏳ Progress: 0%
⏳ Progress: 5%
⏳ Progress: 10%
⏳ Progress: 15%
⏳ Progress: 20%
⏳ Progress: 25%
⏳ Progress: 30%
⏳ Progress: 35%
⏳ Progress: 40%
⏳ Progress: 45%
⏳ Progress: 50%
⏳ Progress: 55%
⏳ Progress: 60%
⏳ Progress: 65%
⏳ Progress: 70%
⏳ Progress: 75%
⏳ Progress: 80%
⏳ Progress: 85%
⏳ Progress: 90%
⏳ Progress: 95%
⏳ Progress: 100%
✅ Demo video ready in 64.60s -> /content/output_demo_(Final).mp4


In [ ]:
import shutil
import os

# مسیر فایل در محیط کولب
local_output = '/content/output_demo_(Final).mp4'

# مسیر مقصد در گوگل درایو
drive_dest = '/content/drive/My Drive/output_demo_final_saved.mp4'

if os.path.exists(local_output):
    shutil.copy(local_output, drive_dest)
    print(f"✅ Video successfully saved to Google Drive at: {drive_dest}")
else:
    print("❌ Local output file not found. Please run the video processing cell first.")

✅ Video successfully saved to Google Drive at: /content/drive/My Drive/output_demo_final_saved.mp4
